In [6]:
import yfinance as yf
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# 1. Загрузка данных
tickers = ["AAPL", "TSLA"]

raw_train = yf.download(
    tickers, start="2014-12-01", end="2025-01-01",
    interval="1mo", auto_adjust=True, progress=False
)["Close"]

raw_test = yf.download(
    tickers, start="2025-01-01", end="2026-01-01",
    interval="1mo", auto_adjust=True, progress=False
)["Close"]

In [7]:
# 2. Оценка параметров на обучающей выборке (лог-returns)
train_returns = np.log(raw_train / raw_train.shift(1)).dropna()

mu    = train_returns.mean().values   # среднемесячные лог-доходности
Sigma = train_returns.cov().values    # ковариационная матрица
rf_monthly = 0.035 / 12              # 3.5% годовых → месячная ставка

print("=" * 55)
print("ОБУЧАЮЩАЯ ВЫБОРКА (2015-01 — 2024-12)")
print(f"  Кол-во месяцев       : {len(train_returns)}")
print(f"  mu  (AAPL, TSLA)     : {mu.round(5)}")
print(f"  σ   (AAPL, TSLA)     : {np.sqrt(np.diag(Sigma)).round(5)}")
print(f"  rf_monthly           : {rf_monthly:.6f}")

ОБУЧАЮЩАЯ ВЫБОРКА (2015-01 — 2024-12)
  Кол-во месяцев       : 120
  mu  (AAPL, TSLA)     : [0.01934 0.02754]
  σ   (AAPL, TSLA)     : [0.07841 0.1661 ]
  rf_monthly           : 0.002917


In [8]:
# 3. Критерий Келли
excess  = mu - rf_monthly
w_kelly = np.linalg.solve(Sigma, excess)
w0      = 1.0 - w_kelly.sum()

# Проверка через BFGS
def neg_kelly(w, mu, Sigma, rf):
    rp  = rf + np.dot(w, mu - rf)
    var = float(w @ Sigma @ w)
    return -(rp - 0.5 * var)

res = minimize(neg_kelly, w_kelly, args=(mu, Sigma, rf_monthly), method="BFGS")
if not res.success:
    print(f"  [Предупреждение] BFGS не сошёлся: {res.message}")
assert np.allclose(res.x, w_kelly, atol=1e-4), "BFGS и аналитика расходятся!"

print()
print("ОПТИМАЛЬНЫЙ ПОРТФЕЛЬ КЕЛЛИ")
print(f"  w_AAPL : {w_kelly[0]:+.4f}  ({w_kelly[0]*100:+.2f}%)")
print(f"  w_TSLA : {w_kelly[1]:+.4f}  ({w_kelly[1]*100:+.2f}%)")
print(f"  w_rf   : {w0:+.4f}  ({w0*100:+.2f}%)")


ОПТИМАЛЬНЫЙ ПОРТФЕЛЬ КЕЛЛИ
  w_AAPL : +2.3143  (+231.43%)
  w_TSLA : +0.3255  (+32.55%)
  w_rf   : -1.6397  (-163.97%)


In [9]:
# 4. Бэктест (2025)
last_train_row = raw_train.iloc[[-1]]              # декабрь 2024
test_prices    = pd.concat([last_train_row, raw_test])
test_returns   = np.log(test_prices / test_prices.shift(1)).dropna()

# Месячная доходность портфеля = доходность рисковой части + безрисковая часть
port_returns = test_returns.values @ w_kelly + w0 * rf_monthly

# Кумулятивная доходность через лог-returns
n_months      = len(port_returns)
cum_log       = port_returns.sum()
norm_return   = np.exp(cum_log) - 1

# Аннуализированная доходность
annual_return = np.exp(cum_log * 12 / n_months) - 1

# Sharpe
mean_excess   = port_returns.mean() - rf_monthly
sharpe        = mean_excess / port_returns.std(ddof=1) * np.sqrt(12)

print()
print("БЭКТЕСТ (2025)")
print(f"  Кол-во месяцев              : {n_months}")
print(f"  Норма доходности (total)    : {norm_return*100:+.2f}%")
print(f"  Аннуализированная доходность: {annual_return*100:+.2f}%")
print(f"  Sharpe Ratio (annualized)   : {sharpe:.4f}")


БЭКТЕСТ (2025)
  Кол-во месяцев              : 12
  Норма доходности (total)    : +19.50%
  Аннуализированная доходность: +19.50%
  Sharpe Ratio (annualized)   : 0.2567
